# Data wrangling for pMTG meanFC cognitive analyses


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression


In [ ]:
ABCD_DATA_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/abcd-data-release-5.1/core')
OUTPUT_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final')
MEANFC_SOURCE_PATH = OUTPUT_DIR / 'pMTG_FC_profiles_midb61_meanFC.csv'
OUTPUT_PATH = OUTPUT_DIR / 'wrangled_pMTG_FC_data_midb61_meanFC.csv'
MOTION_QA_PATH = OUTPUT_DIR / 'motion_QA_results.csv'

BASELINE_EVENT = 'baseline_year_1_arm_1'
ONE_YEAR_EVENT = '1_year_follow_up_y_arm_1'
TWO_YEAR_EVENT = '2_year_follow_up_y_arm_1'
FAMILY_SELECTION_SEED = 42

INVALID_CODES = (555, 777, 888, 999)

INCOME_MEDIANS = {
    1.0: 2500,
    2.0: 8500,
    3.0: 14000,
    4.0: 20500,
    5.0: 30000,
    6.0: 42500,
    7.0: 62500,
    8.0: 87500,
    9.0: 150000,
    10.0: 250000,
}

POVERTY_LINES_2017 = {
    1.0: 12060,
    2.0: 16240,
    3.0: 20420,
    4.0: 24600,
    5.0: 28780,
    6.0: 32960,
    7.0: 37140,
    8.0: 41320,
}

STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b'
}


In [ ]:



def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def read_event_data(relative_path, columns, eventname):
    data = pd.read_csv(ABCD_DATA_DIR / relative_path, usecols=columns)
    data = data[data['eventname'] == eventname].copy()
    data = data.drop(columns=['eventname'])
    return data.drop_duplicates(subset=['src_subject_id'])


def preprocess_ravlt_data(ravlt_data):
    trial_columns = [
        'pea_ravlt_sd_trial_i_tc',
        'pea_ravlt_sd_trial_ii_tc',
        'pea_ravlt_sd_trial_iii_tc',
        'pea_ravlt_sd_trial_iv_tc',
        'pea_ravlt_sd_trial_v_tc',
    ]
    required_columns = [
        'src_subject_id',
        *trial_columns,
        'pea_ravlt_sd_trial_vi_tc',
        'pea_ravlt_ld_trial_vii_tc',
    ]

    ravlt_data = ravlt_data.copy()
    score_columns = trial_columns + [
        'pea_ravlt_sd_trial_vi_tc',
        'pea_ravlt_ld_trial_vii_tc',
    ]
    ravlt_data[score_columns] = ravlt_data[score_columns].apply(
        pd.to_numeric,
        errors='coerce',
    )
    ravlt_data = ravlt_data.dropna(subset=trial_columns).copy()

    ravlt_data['ravlt_learning'] = (
        ravlt_data['pea_ravlt_sd_trial_v_tc']
        - ravlt_data['pea_ravlt_sd_trial_i_tc']
    )
    ravlt_data['ravlt_learningslope'] = ravlt_data[trial_columns].apply(
        lambda row: np.polyfit(np.arange(1, 6), row.to_numpy(dtype=float), 1)[0],
        axis=1,
    )
    ravlt_data['ravlt_immediate'] = ravlt_data[trial_columns].sum(axis=1)
    ravlt_data['ravlt_short_delay'] = ravlt_data['pea_ravlt_sd_trial_vi_tc']
    ravlt_data['ravlt_long_delay'] = ravlt_data['pea_ravlt_ld_trial_vii_tc']

    ravlt_data.replace([np.inf, -np.inf], np.nan, inplace=True)

    return ravlt_data[
        [
            'src_subject_id',
            'ravlt_immediate',
            'ravlt_short_delay',
            'ravlt_long_delay',
        ]
    ]


def get_tanner_stage(row):
    if pd.notna(row['pds_y_ss_female_category_2']) and row['pds_sex_y'] == 2:
        return row['pds_y_ss_female_category_2']
    if pd.notna(row['pds_y_ss_male_cat_2']) and row['pds_sex_y'] == 1:
        return row['pds_y_ss_male_cat_2']
    return np.nan


def get_poverty_line(household_size, poverty_lines=POVERTY_LINES_2017):
    if pd.isna(household_size) or household_size in INVALID_CODES or household_size < 1:
        return np.nan
    if household_size <= 8:
        return poverty_lines[float(int(household_size))]

    extra = (int(household_size) - 8) * 4180
    return poverty_lines[8.0] + extra


def calculate_income_to_needs(
    df,
    income_col='demo_comb_income_v2',
    household_size_col='demo_roster_v2',
    output_col='inr',
    missing_indicator_col=None,
):
    df = df.copy()
    missing_indicator_col = missing_indicator_col or f'{output_col}_missing'

    income = pd.to_numeric(df[income_col], errors='coerce').replace(
        list(INVALID_CODES),
        np.nan,
    )
    household_size = pd.to_numeric(df[household_size_col], errors='coerce').replace(
        list(INVALID_CODES),
        np.nan,
    )

    df['income_median'] = income.map(INCOME_MEDIANS)
    df['poverty_line_2017'] = household_size.apply(get_poverty_line)
    df[output_col] = df['income_median'] / df['poverty_line_2017']
    df[output_col] = df[output_col].replace([np.inf, -np.inf], np.nan)
    df[missing_indicator_col] = df[output_col].isna()
    return df

def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)

    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])

    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)

    return merged


def select_one_per_family(df, family_col='rel_family_id', seed=FAMILY_SELECTION_SEED):
    df = df.copy()
    missing_family = df[family_col].isna()
    selected_family_rows = (
        df.loc[~missing_family]
        .groupby(family_col, group_keys=False)
        .sample(n=1, random_state=seed)
    )
    selected = pd.concat([selected_family_rows, df.loc[missing_family]], axis=0)
    return selected.sort_index().reset_index(drop=True)


def complete_covariate_mask(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete


def encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(df, fc_columns, covariates=STANDARD_FC_COVARIATES):
    df = df.copy()
    covariate_complete = complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df


In [ ]:
abcd_img = read_event_data(
    'imaging/mri_y_adm_info.csv',
    [
        'src_subject_id',
        'eventname',
        'mri_info_softwareversion',
        'mri_info_studydate',
    ],
    TWO_YEAR_EVENT,
)

fam_data = read_event_data(
    'abcd-general/abcd_y_lt.csv',
    ['src_subject_id', 'eventname', 'rel_family_id'],
    BASELINE_EVENT,
)

site_age_data = read_event_data(
    'abcd-general/abcd_y_lt.csv',
    ['src_subject_id', 'eventname', 'site_id_l', 'interview_age'],
    TWO_YEAR_EVENT,
)

hand_data = read_event_data(
    'neurocognition/nc_y_ehis.csv',
    [
        'src_subject_id',
        'eventname',
        'ehi1b',
        'ehi2b',
        'ehi3b',
        'ehi4b',
        'ehi_y_ss_scoreb',
    ],
    BASELINE_EVENT,
)

sex_data = read_event_data(
    'gender-identity-sexual-health/gish_p_gi.csv',
    ['src_subject_id', 'eventname', 'demo_sex_v2'],
    BASELINE_EVENT,
)

print('Scanner/demographic tables loaded:')
for name, table in {
    'abcd_img': abcd_img,
    'fam_data': fam_data,
    'site_age_data': site_age_data,
    'hand_data': hand_data,
    'sex_data': sex_data,
}.items():
    print(f'{name}: {table.shape}')


In [ ]:
nihtb_data_2y = read_event_data(
    'neurocognition/nc_y_nihtb.csv',
    [
        'src_subject_id',
        'eventname',
        'nihtbx_picvocab_uncorrected',
        'nihtbx_picvocab_agecorrected',
        'nihtbx_picvocab_v',
        'nihtbx_flanker_uncorrected',
        'nihtbx_flanker_agecorrected',
        'nihtbx_flanker_v',
        'nihtbx_pattern_uncorrected',
        'nihtbx_pattern_agecorrected',
        'nihtbx_pattern_v',
        'nihtbx_picture_uncorrected',
        'nihtbx_picture_agecorrected',
        'nihtbx_picture_v',
        'nihtbx_reading_uncorrected',
        'nihtbx_reading_agecorrected',
        'nihtbx_reading_v',
        'nihtbx_picvocab_fc',
        'nihtbx_flanker_fc',
        'nihtbx_pattern_fc',
        'nihtbx_picture_fc',
        'nihtbx_reading_fc',
    ],
    TWO_YEAR_EVENT,
)
nihtb_data_2y = nihtb_data_2y.rename(
    columns={col: f'{col}_2y' for col in nihtb_data_2y.columns if col != 'src_subject_id'}
)

ravlt_data = read_event_data(
    'neurocognition/nc_y_ravlt.csv',
    [
        'src_subject_id',
        'eventname',
        'pea_ravlt_sd_trial_i_tc',
        'pea_ravlt_sd_trial_ii_tc',
        'pea_ravlt_sd_trial_iii_tc',
        'pea_ravlt_sd_trial_iv_tc',
        'pea_ravlt_sd_trial_v_tc',
        'pea_ravlt_sd_listb_tc',
        'pea_ravlt_sd_trial_vi_tc',
        'pea_ravlt_ld_trial_vii_tc',
    ],
    TWO_YEAR_EVENT,
)
ravlt_summary = preprocess_ravlt_data(ravlt_data)
ravlt_summary = ravlt_summary.rename(
    columns={col: f'{col}_2y' for col in ravlt_summary.columns if col != 'src_subject_id'}
)

ses_data = read_event_data(
    'abcd-general/abcd_p_demo.csv',
    [
        'src_subject_id',
        'eventname',
        'demo_comb_income_v2',
        'demo_roster_v2',
        'demo_prnt_ed_v2_2yr_l',
        'demo_prtnr_ed_v2_2yr_l',
        'demo_prnt_marital_v2',
        'demo_prnt_prtnr_bio',
    ],
    BASELINE_EVENT,
)

bilingual_data = read_event_data(
    'culture-environment/ce_y_acc.csv',
    ['src_subject_id', 'eventname', 'accult_q2_y'],
    BASELINE_EVENT,
)

dual_lang = read_event_data(
    'abcd-general/abcd_p_demo.csv',
    ['src_subject_id', 'eventname', 'demo_dual_lang_v2_l'],
    ONE_YEAR_EVENT,
)

flanker_screen = read_event_data(
    'neurocognition/nc_y_flkr.csv',
    [
        'src_subject_id',
        'eventname',
        'flkr_scr_trialcount',
        'flkr_scr_propcorrect',
        'flkr_scr_meanrt',
        'flkr_scr_trialcount_congruent',
        'flkr_scr_propcorrect_congruent',
        'flkr_scr_meanrt_congruent',
        'flkr_scr_medrt_congruent',
        'flkr_scr_trialcount_incongruent',
        'flkr_scr_propcorrect_incongruent',
        'flkr_scr_meanrt_incongruent',
        'flkr_scr_medrt_incongruent',
    ],
    TWO_YEAR_EVENT,
)
flanker_screen = flanker_screen.rename(
    columns={col: f'{col}_2y' for col in flanker_screen.columns if col != 'src_subject_id'}
)

sst_data = read_event_data(
    'imaging/mri_y_tfmr_sst_beh.csv',
    ['src_subject_id', 'eventname', 'tfmri_sst_all_beh_total_issrt'],
    TWO_YEAR_EVENT,
)

print('Cognitive, SES, language, Flanker, and SST tables loaded:')
for name, table in {
    'nihtb_data_2y': nihtb_data_2y,
    'ravlt_summary': ravlt_summary,
    'ses_data': ses_data,
    'bilingual_data': bilingual_data,
    'dual_lang': dual_lang,
    'flanker_screen': flanker_screen,
    'sst_data': sst_data,
}.items():
    print(f'{name}: {table.shape}')


In [ ]:
motion_df = read_event_data(
    'imaging/mri_y_qc_motion.csv',
    [
        'src_subject_id',
        'eventname',
        'rsfmri_meanmotion',
        'rsfmri_maxmotion',
        'rsfmri_ntpoints',
        'rsfmri_nvols',
        'rsfmri_numtrs',
    ],
    TWO_YEAR_EVENT,
)

pds_full = pd.read_csv(ABCD_DATA_DIR / 'physical-health/ph_y_pds.csv')
baseline_sex = pds_full[pds_full['eventname'] == BASELINE_EVENT][
    ['src_subject_id', 'pds_sex_y']
].drop_duplicates(subset=['src_subject_id'])
tanner_df = pds_full[pds_full['eventname'] == TWO_YEAR_EVENT][
    [
        'src_subject_id',
        'pds_bdyhair_y',
        'pds_f4_2_y',
        'pds_f5_y',
        'pds_m4_y',
        'pds_m5_y',
        'pds_y_ss_female_category_2',
        'pds_y_ss_male_cat_2',
    ]
].drop_duplicates(subset=['src_subject_id'])
tanner_df = tanner_df.merge(baseline_sex, on='src_subject_id', how='left')
tanner_df['tanner_stage'] = tanner_df.apply(get_tanner_stage, axis=1)

qa_df = read_event_data(
    'imaging/mri_y_qc_incl.csv',
    ['src_subject_id', 'eventname', 'imgincl_rsfmri_include'],
    TWO_YEAR_EVENT,
)

demo_df = read_event_data(
    'abcd-general/abcd_p_demo.csv',
    [
        'src_subject_id',
        'eventname',
        'demo_race_a_p___10',
        'demo_race_a_p___11',
        'demo_race_a_p___12',
        'demo_race_a_p___13',
        'demo_race_a_p___14',
        'demo_race_a_p___15',
        'demo_race_a_p___16',
        'demo_race_a_p___17',
        'demo_race_a_p___18',
        'demo_race_a_p___19',
        'demo_race_a_p___20',
        'demo_race_a_p___21',
        'demo_race_a_p___22',
        'demo_race_a_p___23',
        'demo_race_a_p___24',
        'demo_race_a_p___25',
        'demo_race_a_p___77',
        'demo_race_a_p___99',
        'demo_ethn_v2',
        'demo_ethn2_v2',
    ],
    BASELINE_EVENT,
)

print('Motion, Tanner, QA, and race/ethnicity tables loaded:')
for name, table in {
    'motion_df': motion_df,
    'tanner_df': tanner_df,
    'qa_df': qa_df,
    'demo_df': demo_df,
}.items():
    print(f'{name}: {table.shape}')


In [ ]:
combined_df = (
    abcd_img.merge(fam_data, on='src_subject_id')
    .merge(site_age_data, on='src_subject_id')
    .merge(hand_data, on='src_subject_id')
    .merge(sex_data, on='src_subject_id')
    .merge(nihtb_data_2y, on='src_subject_id', how='left')
    .merge(ravlt_summary, on='src_subject_id', how='left')
    .merge(ses_data, on='src_subject_id', how='left')
    .merge(bilingual_data, on='src_subject_id', how='left')
    .merge(motion_df, on='src_subject_id', how='left')
    .merge(tanner_df, on='src_subject_id', how='left')
    .merge(sst_data, on='src_subject_id', how='left')
    .merge(qa_df, on='src_subject_id', how='left')
    .merge(demo_df, on='src_subject_id', how='left')
    .merge(dual_lang, on='src_subject_id', how='left')
    .merge(flanker_screen, on='src_subject_id', how='left')
    .drop_duplicates(subset=['src_subject_id'])
)

combined_df['src_subject_id'] = standardize_subject_id(combined_df['src_subject_id'])
combined_df = calculate_income_to_needs(combined_df)

print(f'Combined ABCD data shape before FC merge: {combined_df.shape}')
print(f'Missing raw INR values: {combined_df["inr"].isna().sum()}')
display(combined_df.head())


In [ ]:
meanfc_source = pd.read_csv(MEANFC_SOURCE_PATH)
if 'subject_id' in meanfc_source.columns:
    meanfc_source = meanfc_source.rename(columns={'subject_id': 'src_subject_id'})

meanfc_source['src_subject_id'] = standardize_subject_id(meanfc_source['src_subject_id'])
raw_fc_cols = [
    col for col in meanfc_source.columns
    if col.endswith('_fz') and not col.endswith('_resid') and '_full' not in col
]
full_fc_cols = [
    col for col in meanfc_source.columns
    if '_fz' in str(col) and '_full' in str(col)
]
print(f'Generated full-network Fisher-z FC columns excluded from analysis table: {len(full_fc_cols)}')

derived_cols = [
    col for col in [
        'latent_factor_ss_general_ses',
        'latent_factor_ss_social',
        'latent_factor_ss_perinatal',
    ]
    if col in meanfc_source.columns
]
meanfc_keep_cols = ['src_subject_id', *raw_fc_cols, *derived_cols]
conn_df = meanfc_source[meanfc_keep_cols].copy()
conn_df = conn_df.dropna(subset=raw_fc_cols)
conn_df = conn_df.drop_duplicates(subset=['src_subject_id'])

merged_data = combined_df.merge(conn_df, on='src_subject_id', how='right')
merged_data = merge_motion_qa(merged_data, MOTION_QA_PATH, how='left')

n_before_qa = len(merged_data)
merged_data = merged_data[merged_data['imgincl_rsfmri_include'].eq(1)].copy()
print(f'Rows retained after rsfMRI inclusion filter: {len(merged_data)}/{n_before_qa}')

n_before_family_selection = len(merged_data)
merged_data = select_one_per_family(
    merged_data,
    family_col='rel_family_id',
    seed=FAMILY_SELECTION_SEED,
)
print(
    'Rows retained after random one-participant-per-family selection: '
    f'{len(merged_data)}/{n_before_family_selection}'
)
print(
    'Families with more than one retained participant: '
    f'{merged_data["rel_family_id"].dropna().duplicated().sum()}'
)

print(f'Loaded raw meanFC columns: {len(raw_fc_cols)}')
print(f'Preserved derived latent factor columns from Final meanFC table: {derived_cols}')
print(f'Merged ABCD + meanFC data shape: {merged_data.shape}')
print(f'Missing mean_fd_0.20 values after motion QA merge: {merged_data["mean_fd_0.20"].isna().sum()}')
display(merged_data.head())


In [ ]:
existing_resid_cols = [f'{col}_resid' for col in raw_fc_cols if f'{col}_resid' in merged_data.columns]
if existing_resid_cols:
    merged_data = merged_data.drop(columns=existing_resid_cols)

print('Residualizing meanFC columns for:', STANDARD_FC_COVARIATES)
merged_data = residualize_fc_profiles(
    merged_data,
    raw_fc_cols,
    covariates=STANDARD_FC_COVARIATES,
)

fc_resid_cols = [f'{col}_resid' for col in raw_fc_cols]
print(f'Created residualized meanFC columns: {len(fc_resid_cols)}')


In [ ]:
merged_data.to_csv(OUTPUT_PATH, index=False)
print(f'Saved wrangled ABCD + pMTG meanFC data to {OUTPUT_PATH}')
print(f'Final shape: {merged_data.shape}')


In [ ]:
demographic_summary_cols = [
    'src_subject_id',
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'rel_family_id',
    'demo_ethn_v2',
    'demo_ethn2_v2',
    'ehi1b',
    'inr',
    'inr_missing',
    'mean_fd_0.20',
]
available_summary_cols = [col for col in demographic_summary_cols if col in merged_data.columns]
display(merged_data[available_summary_cols].head())

print('Final columns:')
for column in merged_data.columns:
    print(column)
